# Dokumente laden

Alle PDF dokumente aus dem `data` Ordner werden geladen. Der Inhalt wird dabei vom restlichen Text getrennt. Für den [OpenDataLoader](https://github.com/opendataloader-project/opendataloader-pdf) muss Java installiert sein.

In [ ]:
from langchain_core.documents import Document
from langchain_opendataloader_pdf import OpenDataLoaderPDFLoader
from website_crawler import WebsiteCrawler
import glob

loader = OpenDataLoaderPDFLoader(
    file_path=glob.glob("data/*.pdf"),
    format="markdown"
)
docs = loader.load()

WEB_START_URLS = [
    "https://www.th-koeln.de/studium/informatik-und-systems-engineering-bachelor_126263.php"
]
docs.extend(WebsiteCrawler(max_pages=20, max_depth=1).crawl(WEB_START_URLS))

def print_doc(doc: Document):
    print(f"---\n{doc.page_content}\n---")
    if "source" in doc.metadata:
        print(f"Quelle: {doc.metadata["source"]}, ")
    if "page" in doc.metadata:
        print(f"Seite {doc.metadata["page"]}")

for doc in docs:
    print_doc(doc)


# Indexing
Der Inhalt aus den Dokumenten wird in Abschnitte unterteilt. Diese Abschnitte werden in hochdimensionale Vektoren kodiert, sodass der Text unabhängig von bestimmten Schlagwörtern nach relevanten Passagen zu einer Frage durchsucht werden kann.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()  # Load API keys

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs)

vectorstore = InMemoryVectorStore.from_documents(
    documents=all_splits,
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


# LLM und Prompt-Generierung

Aus dem Vektor Store werden zu jedem Prompt relevante Textstellen gefunden und angehängt, damit das LLM anhand dieser Textstellen die Frage beantworten kann. Der Prompt aus Frage und Textstellen wird dann an einen LLM Provider gesendet. Die Antwort des LLMs wird zusammen mit den gefundenen Textstellen zurückgegeben.

In [ ]:
from langsmith import traceable
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini", temperature=1)


# Add decorator so this function is traced in LangSmith
@traceable()
def rag_bot(question: str) -> dict:
    # LangChain retriever will be automatically traced
    docs = retriever.invoke(question)
    docs_string = "".join(doc.page_content for doc in docs)
    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions. Use the following source documents to answer the user's questions. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise. <context>{docs_string}</context>"""
    # langchain ChatModel will be automatically traced
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ],
    )
    return {"answer": ai_msg.content, "documents": docs}

# Manueller Test

Das RAG wird mit einer einzelnen Frage getestet. Die Frage kann bearbeitet und die Codezelle erneut ausgeführt werden, um verschiedene Fragen zu stellen.

In [ ]:
question = "Ich interessiere mich nicht nur für Softwareentwicklung, sondern auch für Management und Organisation von Softwareprojekten. Deckt der Studiengang Informatik und Systems-Engineering dieses Themengebiet auch ab?"
response = rag_bot(question)
print(response["answer"])
if "documents" in response and response["documents"]:
    doc = response["documents"][0]
    print(f"Relevanteste Textpassage:\n---\n{doc.page_content}\n---")
    if "source" in doc.metadata:
        print(f"Quelle: {doc.metadata["source"]}, ")
    if "page" in doc.metadata:
        print(f"Seite {doc.metadata["page"]}")


# Automatisierter Test

Das RAG wird mit einem Datensatz aus Fragen getestet. Ein LLM beurteilt die Qualität der Antworten auf Basis der Musterantworten.

In [ ]:
from langsmith import Client
from typing_extensions import Annotated, TypedDict
from langchain_openai import ChatOpenAI

client = Client()

examples = [
    {
        "inputs": {"question": "Wird das Programmierpraktikum benotet?"},
        "outputs": {"answer": "Nein, das Modul 'Programmierpraktikum' wird nicht benotet."},
    },
    {
        "inputs": {"question": "Wie viele Versuche habe ich, um eine Prüfung zu bestehen?"},
        "outputs": {
            "answer": "Zu jedem Modul haben Sie 3 Prüfungsversuche. Ausschließlich an der TH Köln bekommen Studierende vier zusätzliche Prüfungsversuche für das gesamte Studium."},
    },
    {
        "inputs": {"question": "Wie viele Wochenstunden sollte ich für das Modul Mathematik 2 einplanen?"},
        "outputs": {"answer": "Das Modul Mathematik 2 hat 8 Semesterwochenstunden."},
    },
    {
        "inputs": {"question": "Wann finden die Prüfungen statt?"},
        "outputs": {"answer": "Die Prüfungen finden in der Prüfungszeit vom 9. Februar bis zum 13. März 2026 statt."},
    },
    {
        "inputs": {
            "question": "Ich kann das Modul IT-Projekt-Management in PSSO nicht finden. Wie soll ich mich anmelden?"},
        "outputs": {
            "answer": " Die Anmeldung für dieses Modul erfolgt ausnahmnsweise nicht über PSSO, sondern über in ILU im Ordner von Frau Prof. Yuan für den Kurs IT-Projekt-Management (IPM)."},
    },
    {
        "inputs": {"question": "Gib mir die Kontaktinformationen der Studiengangsberatung."}, "outputs": {
        "answer": "Die Studiengangsberatung ist Prof. Dr. Beate Rhein. Ihr Büro ist in Raum ZN-06-05. Sie ist telefonisch unter der Nummer 0221/8275-2291 und über die Mailadresse studiengangleitung-batin@th-koeln.de erreichbar."},
    },
    {
        "inputs": {
            "question": "Kann ich noch von Technische Informatik zu Informatik und Systems-Engineering wechseln, wenn ich meine Abschlussarbeit schon angemeldet habe?"},
        "outputs": {"answer": "Nein, ein Wechsel nach der Anmeldung der Bachelorarbeit ist nicht möglich."},
    },
    {
        "inputs": {"question": "Brauche ich ein eigenes Laptop? Welche Anforderungen muss es erfüllen?"},
        "outputs": {"answer": "Ein aktuelles Laptop mit 4-8 GB Arbeitsspeicher und Windows oder Linux wird empfohlen."},
    },
    {
        "inputs": {"question": "Welche Studienschwerpunkte gibt es in Informatik und Systems-Engineering?"},
        "outputs": {
            "answer": "In Informatik und Systems-Engineering gibt es die Schwerpunkte Künstliche Intelligenz, Technische Informatik und Verteilte Software-Systeme."},
    },
    {
        "inputs": {
            "question": "Welche Unterlagen werden benötigt, um Leistungen aus einem anderen Studiengang anerkennen zu lassen?"},
        "outputs": {
            "answer": "Um die Anerkennung zu prüfen braucht die Studiengangsleitung ihren Notenspiegel der vorherigen Hochschule, Modulbeschreibungen aller anzuerkennenden Module und eine Angabe, welches Modul des vorherigen Studiengangs auf welches Modul hier anerkannt werden soll."},
    },
    {
        "inputs": {
            "question": "Ich interessiere mich nicht nur für Softwareentwicklung, sondern auch für Management und Organisation von Softwareprojekten. Deckt der Studiengang Informatik und Systems-Engineering dieses Themengebiet auch ab?"},
        "outputs": {
            "answer": "Ja, diese Themen sind ebenfalls Bestandteil des Studiums. Der Studiengang legt nicht nur Wert auf Fachwissen, sondern auch auf die Ausbildung in Softskills, zu denen u. a. Projektmanagement, Teamarbeit und Präsentationstechniken gehören."},
    },
    {
        "inputs": {
            "question": "Gibt es einen Numerus Clausus oder eine andere Zulassungsbeschränkung?"},
        "outputs": {"answer": "Nein, das Studium ist nicht zulassungsbeschränkt."},
    },
    {
        "inputs": {
            "question": "Wie viele ECTS brauche ich insgesamt, um das Studium abzuschließen?"},
        "outputs": {"answer": "Das Studium hat einen Umfang von 210 ECTS."},
    },
    {
        "inputs": {
            "question": "Kann ich das Studium auch nächstes Jahr im Sommer anfangen?"},
        "outputs": {"answer": "Nein, Studienbeginn ist im Wintersemester."},
    },
    {
        "inputs": {
            "question": "Muss man Deutsch können, um an der TH Köln Informatik und Systems Engineering zu studieren?"},
        "outputs": {
            "answer": "Ja, Deutschkenntnisse sind notwendig, weil die Lehrveranstaltungen auf Deutsch gehalten werden."},
    },
]

dataset_name = "MLWR RAG 3"
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(
        dataset_id=dataset.id,
        examples=examples
    )


# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]


# Grade prompt
correctness_instructions = """You are a teacher grading a quiz. You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. (2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. Avoid simply stating the correct answer at the outset."""

grader_llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)


def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""
    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers}
    ])
    return grade["correct"]


def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])


experiment_results = client.evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness],
    experiment_prefix="mlwr-rag-correctness",
    metadata={"version": "LCEL context, gpt-4-0125-preview"},
)

# Explore results locally as a dataframe if you have pandas installed
# experiment_results.to_pandas()